# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (code only)

Loads the starter CSV, finds the rule's eligible pages, and splits whole clients into dev and test halves.

In [1]:
import os, json, numpy as np, pandas as pd

CSV_NAME = 'content_refresh_anonymized.csv'
RAW_URL  = 'https://raw.githubusercontent.com/stanley-metone/flyrank-ml-internship-starter/main/data/raw/' + CSV_NAME

# Find the repo root (works from the repo root or from work/notebooks/); fall back to GitHub raw.
ROOT, p = None, os.getcwd()
for _ in range(4):
    if os.path.exists(os.path.join(p, 'data', 'raw', CSV_NAME)):
        ROOT = p; break
    p = os.path.dirname(p)
df = pd.read_csv(os.path.join(ROOT, 'data', 'raw', CSV_NAME)) if ROOT else pd.read_csv(RAW_URL)
OUT_DIR = os.path.join(ROOT or os.getcwd(), 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
SEED = 42

# EVALUATION label only. It is never an input to the rule.
df['declined'] = (df['trend_direction'] == 'down').astype(int)

# The label can only be 'down' when the page had impressions in the earlier 30 days.
# Pages with none are 0% 'down' by construction, so they are excluded from checks and scoring.
eligible = df['impressions_prev_30d'] > 0
print(f'rows: {len(df):,} | eligible (impressions_prev_30d > 0): {eligible.sum():,} | '
      f'excluded by construction: {(~eligible).sum():,}')

# Split WHOLE CLIENTS in half. Dev clients: look at tables, choose thresholds.
# Test clients: evaluate the finished rule. Test clients are never used to pick a threshold.
clients = np.array(sorted(df['client_id'].unique()))
np.random.default_rng(SEED).shuffle(clients)
dev_clients = set(clients[: len(clients) // 2])
df['split'] = np.where(df['client_id'].isin(dev_clients), 'dev', 'test')
for s in ('dev', 'test'):
    x = df[eligible & (df['split'] == s)]
    print(f'{s:4} clients={x.client_id.nunique():>2}  eligible rows={len(x):>6,}  share declined={x.declined.mean():.3f}')
dev = df[eligible & (df['split'] == 'dev')]

rows: 30,000 | eligible (impressions_prev_30d > 0): 26,612 | excluded by construction: 3,388
dev  clients=16  eligible rows= 9,508  share declined=0.626
test clients=15  eligible rows=17,104  share declined=0.603


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Lane (provisional, to be confirmed before lanes lock this week):** Refresh / Content Opportunity Scoring.

**The rule in plain words.** Review a page first if it is young (180 days old or less), had a mid-sized audience in the earlier 30 days (200 to 999 impressions), and has gone the longest without an update. The staler page goes first.

**Score:** days since last update, for pages that pass the rule. Zero for everything else.
**Reason code (one):** `young_visible_stale_page`
**Action label:** `REVIEW_FOR_REFRESH`

**What the rule is for.** It ranks pages for a person to look at. It does not say an update would fix a page.

**How I judge it.** The label is `trend_direction == 'down'`: impressions in the last 30 days fell more than 20% against the 30 days before. The label is only used to check the rule. It never goes into it. A page with no impressions in the earlier 30 days cannot be labelled 'down', so those 3,388 pages are left out of the checks and the scoring.

**How I kept myself honest.** I split whole clients into two halves. I read the tables and chose thresholds on the dev half only. I scored the finished rule on the test half.

### Signal check 1 of 2: page age (flag-linked)

FlyRank's `page_one_decay_risk` code treats older pages (180 days or more) as riskier. The table shows the share of pages that declined, by page age. Dev clients only.

In [2]:
MIN_N = 200   # a bucket needs at least this many pages before it counts toward a verdict

def bucket_table(d, col, bins, labels):
    b = pd.cut(d[col], bins=bins, labels=labels, include_lowest=True)
    t = d.groupby(b, observed=True).agg(n=('declined', 'size'), share_declined=('declined', 'mean'))
    t['share_declined'] = t['share_declined'].round(3)
    return t

def verdict(t, expect, min_n=MIN_N, tol=0.03):
    """expect = 'up' if the flag assumes the share declined rises down the table. One word back."""
    r = t[t['n'] >= min_n]['share_declined'].values
    if len(r) < 2: return 'MIXED'
    d = np.diff(r)
    if r.max() - r.min() < tol: return 'FALSE'
    if (d >= -tol / 3).all():  return 'CONFIRMED' if expect == 'up' else 'OPPOSITE'
    if (d <=  tol / 3).all():  return 'CONFIRMED' if expect == 'down' else 'OPPOSITE'
    return 'MIXED'

# ---- Signal 1 (flag-linked): page age. Flag logic `page_one_decay_risk` assumes OLDER = riskier.
age_t = bucket_table(dev, 'content_age_days', [89, 180, 365, 600], ['90-180 days', '181-365 days', '366+ days'])
print('SIGNAL 1: page age, dev clients only')
print(age_t)
print('flag assumes: older pages decline more  ->  VERDICT:', verdict(age_t, expect='up'))

SIGNAL 1: page age, dev clients only
                     n  share_declined
content_age_days                      
90-180 days       3868           0.718
181-365 days      3049           0.665
366+ days         2591           0.442
flag assumes: older pages decline more  ->  VERDICT: OPPOSITE


**Verdict: OPPOSITE.** The flag assumes older pages decline more. Here the youngest pages declined most: 71.8% at 90-180 days, 66.5% at 181-365 days and 44.2% at 366+ days (n in the table above).

This is a clearly explained negative. A rule built on "older means riskier" would rank the wrong end of the table. One possible reason, not tested here: new pages often gain impressions early and then settle back, so a 20% fall is easier to hit from a launch peak. This table cannot tell me whether that is the cause.

### Signal check 2 of 2: staleness (flag-linked)

FlyRank's `stale_visible_page` code treats pages not updated for 180+ days as riskier. Almost no page in this slice has gone that long, so the table cannot test the 180-day cutoff. It can test whether more days without an update goes with more decline. Buckets under n=200 do not count toward the verdict.

In [3]:
# ---- Signal 2 (flag-linked): staleness. Flag logic `stale_visible_page` assumes LONGER since update = riskier.
stale_t = bucket_table(dev, 'days_since_last_update', [-1, 30, 90, 180, 400],
                       ['0-30 days', '31-90 days', '91-180 days', '181+ days'])
print('SIGNAL 2: days since last update, dev clients only (buckets under n=200 do not count)')
print(stale_t)
print('flag assumes: staler pages decline more  ->  VERDICT (raw):', verdict(stale_t, expect='up'))

# Age and staleness are tangled (young pages are often the freshly updated ones). Does staleness still
# matter inside each age band? Compare stale (>= 90 days) vs not, per band.
dev2 = dev.assign(age_band=pd.cut(dev['content_age_days'], [89, 180, 365, 600],
                                  labels=['90-180 days', '181-365 days', '366+ days']),
                  stale=np.where(dev['days_since_last_update'] >= 90, 'stale (>=90d)', 'recent (<90d)'))
within = (dev2.groupby(['age_band', 'stale'], observed=True)
              .agg(n=('declined', 'size'), share_declined=('declined', 'mean')).round(3))
print('\nStaleness inside each age band (dev clients):')
print(within)

SIGNAL 2: days since last update, dev clients only (buckets under n=200 do not count)
                           n  share_declined
days_since_last_update                      
0-30 days               7223           0.590
31-90 days               123           0.610
91-180 days             2106           0.751
181+ days                 56           0.554
flag assumes: staler pages decline more  ->  VERDICT (raw): CONFIRMED



Staleness inside each age band (dev clients):
                               n  share_declined
age_band     stale                              
90-180 days  recent (<90d)  3389           0.697
             stale (>=90d)   479           0.866
181-365 days recent (<90d)  1887           0.585
             stale (>=90d)  1162           0.796
366+ days    recent (<90d)  2070           0.422
             stale (>=90d)   521           0.522


**Verdict: CONFIRMED, with limits.** Pages that went 91-180 days without an update declined more (75.1%, n=2,106) than pages updated in the last 30 days (59.0%, n=7,223).

Age and staleness are tangled, so I checked inside each age band. The stale group declined more in all three: 86.6% vs 69.7%, 79.6% vs 58.5%, and 52.2% vs 42.2%. The result does not come from age alone.

Two limits. The 181+ day bucket has only 56 pages, so FlyRank's 180-day cutoff is untested here. And an update inside the last 30 days could be a reaction to a decline, so "recently updated" pages may differ for a reason I cannot see.

### Volume band the rule uses

The rule needs an audience band. This table is not one of the two required checks. It is the evidence for the 200-999 band.

In [4]:
# ---- Volume band used by the rule: share declined by impressions in the earlier 30 days (dev clients).
vol_t = bucket_table(dev, 'impressions_prev_30d', [0, 99, 199, 999, 4999, 1e9],
                     ['1-99', '100-199', '200-999', '1,000-4,999', '5,000+'])
print('VOLUME BAND, dev clients only')
print(vol_t)

VOLUME BAND, dev clients only
                         n  share_declined
impressions_prev_30d                      
1-99                  3244           0.606
100-199               1031           0.672
200-999               2698           0.697
1,000-4,999           1894           0.579
5,000+                 641           0.487


**How the checks became the rule.** Age said: use young pages, the opposite of the flag. Staleness said: among those, put the longest-without-update first. The volume table is mixed. The share declined rises to 69.7% at 200-999 impressions, then falls to 57.9% at 1,000-4,999 and 48.7% at 5,000+. So the rule uses the 200-999 band. A possible reason, not tested: very large pages move less, and very small ones are mostly noise.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score:** days since last update, for pages that pass the rule (young, 200-999 impressions). Zero for everything else. Ties break by earlier-30-day impressions, highest first. Pages that do not pass score zero and carry the action `NO_ACTION`. The CSV holds all 30,000 pages. It stays out of git, and this notebook rewrites it on every run.

In [5]:
# THE RULE (inputs: content_age_days, impressions_prev_30d, days_since_last_update. All known before the
# outcome window. No trend_direction, no trend_pct, no *_last_30d, no 90-day totals.)
AGE_MAX, VOL_LO, VOL_HI = 180, 200, 1000   # thresholds read off the dev-client tables above

young   = df['content_age_days'] <= AGE_MAX
mid_vis = (df['impressions_prev_30d'] >= VOL_LO) & (df['impressions_prev_30d'] < VOL_HI)
hit     = young & mid_vis

# Score: staler first among pages that pass the rule. Zero for pages that do not.
df['score']        = np.where(hit, df['days_since_last_update'], 0)
df['reason_code']  = np.where(hit, 'young_visible_stale_page', 'none')
df['action_label'] = np.where(hit, 'REVIEW_FOR_REFRESH', 'NO_ACTION')

queue = (df.sort_values(['score', 'impressions_prev_30d'], ascending=[False, False], kind='stable')
           .reset_index(drop=True))
queue['rank'] = np.arange(1, len(queue) + 1)
cols = ['rank', 'content_id', 'client_id', 'score', 'reason_code', 'action_label',
        'content_age_days', 'days_since_last_update', 'impressions_prev_30d']
queue[cols].to_csv(os.path.join(OUT_DIR, 'baseline_action_score.csv'), index=False)

print(f'pages passing the rule: {hit.sum():,} of {len(df):,}')
print('wrote work/outputs/baseline_action_score.csv')
queue[cols].head(12)

pages passing the rule: 2,934 of 30,000
wrote work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action_label,content_age_days,days_since_last_update,impressions_prev_30d
0,1,content_c3970ef960d5,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,990
1,2,content_69fad7e6c50c,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,951
2,3,content_7bba5d83959a,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,927
3,4,content_b08562686d22,client_f369cb89fc,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,923
4,5,content_9b28cf5ae4f3,client_f369cb89fc,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,891
5,6,content_452a4e18212c,client_f369cb89fc,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,888
6,7,content_5aa14ed789b2,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,871
7,8,content_701f7861d94a,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,792
8,9,content_434d25702cdb,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,725
9,10,content_e91ce7f0a0b3,client_7f2253d7e2,106,young_visible_stale_page,REVIEW_FOR_REFRESH,106,106,508


**How it is judged.** Precision@K: of the top K pages the rule ranks, what share actually declined. Next to it sit the base rate (the share that declined among all test pages), random picking, and a naive rule, "most visible first". Test clients only. The thresholds came from the dev clients.

In [6]:
def precision_at_k(d, score, k):
    top = d.assign(_s=score).sort_values('_s', ascending=False, kind='stable').head(k)
    return top['declined'].mean()

test = df[eligible & (df['split'] == 'test')]
rng  = np.random.default_rng(SEED)
rows = []
for k in (10, 50, 200):
    rand = np.mean([test.sample(k, random_state=int(s)).declined.mean() for s in rng.integers(0, 10**6, 200)])
    rows.append({'K': k,
                 'my rule': round(precision_at_k(test, test['score'], k), 3),
                 'naive: most visible first': round(precision_at_k(test, test['impressions_prev_30d'], k), 3),
                 'random (mean of 200 draws)': round(rand, 3),
                 'base rate (all test pages)': round(test.declined.mean(), 3)})
ev = pd.DataFrame(rows).set_index('K')
print(f'Precision@K on TEST clients (n eligible pages = {len(test):,}). Higher is better.')
ev

Precision@K on TEST clients (n eligible pages = 17,104). Higher is better.


,my rule,naive: most visible first,random (mean of 200 draws),base rate (all test pages)
K,,,,
10,0.90,0.40,0.622,0.603
50,0.82,0.50,0.608,0.603
200,0.72,0.47,0.604,0.603


**Reading it.** On the test clients the rule gets 0.90 at K=10, 0.82 at K=50 and 0.72 at K=200. Random picking gets about 0.60 to 0.62, and the base rate is 0.603. The naive "most visible first" rule gets 0.40, 0.50 and 0.47, which is worse than random. That is the number a Week 5 model has to beat: **0.82 at K=50**.

Three cautions. (1) The top of the queue is concentrated in a few clients (section 4), so the precision rests on fewer independent bets than K suggests. (2) Six rule variants were compared during development with the test clients in view, so the test numbers are a little optimistic. (3) "Declined" is a proxy: a fall of more than 20% over 30 days.

## 3. Top-10 review

*For each of the top ten: action, why it is there, and what would make it wrong.*

In [7]:
top10 = queue.head(10).copy()

def wrong_if(r):
    n_client = int((top10['client_id'] == r['client_id']).sum())
    bits = [f"one site-wide change on this client's site is the real cause, which would make all {n_client} of its pages in this top 10 wrong together"]
    if r['impressions_prev_30d'] >= 700:
        bits.append(f"its {int(r['impressions_prev_30d'])} impressions in the earlier 30 days were a one-off spike, so a fall is a return to normal")
    else:
        bits.append(f"{int(r['impressions_prev_30d'])} impressions is a small count, so a 20% fall can be a few lost clicks")
    if r['content_age_days'] <= 120:
        bits.append(f"at {int(r['content_age_days'])} days old it may still be settling after launch, not decaying")
    else:
        bits.append("a newer page on the same site absorbed its queries (consolidation), so the page is not stale, just replaced")
    return 'It is wrong if ' + '; or '.join(bits) + '.'

def why(r):
    never = r['content_age_days'] == r['days_since_last_update']
    tail = 'never updated since it was created' if never else f"last updated {int(r['days_since_last_update'])} days ago"
    return (f"{int(r['content_age_days'])}-day-old page with {int(r['impressions_prev_30d'])} impressions "
            f"in the earlier 30 days, {tail}")

top10['action'] = top10['action_label']
top10['why'] = top10.apply(why, axis=1)
top10['what_would_make_it_wrong'] = top10.apply(wrong_if, axis=1)
for _, r in top10.iterrows():
    print(f"#{int(r['rank'])}  {r['content_id']}  [{r['action']}]\n    why: {r['why']}\n    {r['what_would_make_it_wrong']}\n")

#1  content_c3970ef960d5  [REVIEW_FOR_REFRESH]
    why: 106-day-old page with 990 impressions in the earlier 30 days, never updated since it was created
    It is wrong if one site-wide change on this client's site is the real cause, which would make all 7 of its pages in this top 10 wrong together; or its 990 impressions in the earlier 30 days were a one-off spike, so a fall is a return to normal; or at 106 days old it may still be settling after launch, not decaying.

#2  content_69fad7e6c50c  [REVIEW_FOR_REFRESH]
    why: 106-day-old page with 951 impressions in the earlier 30 days, never updated since it was created
    It is wrong if one site-wide change on this client's site is the real cause, which would make all 7 of its pages in this top 10 wrong together; or its 951 impressions in the earlier 30 days were a one-off spike, so a fall is a return to normal; or at 106 days old it may still be settling after launch, not decaying.

#3  content_7bba5d83959a  [REVIEW_FOR_REFRESH]
   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks.** All ten of the top ten declined. I do not count that as ten wins. The ten are one cohort: every page is 106 days old and was never updated since it was created, and they come from two clients, seven from one. "Stale" at the top of this queue means something specific: the page was never updated since it was created. Only 103 of the 26,612 eligible pages are like that, and 86.4% of them declined against 61.0% for the rest. That is a strong signal on a very small group. If something changed on one client's site, those pages would be right or wrong together. The cells below measure how concentrated the queue is and how the rule does by client.

In [8]:
# Outcome check on the top 10 (this peeks at the label to judge the rule, never to build it).
chk = top10[['rank', 'content_id', 'declined']].copy()
chk['outcome'] = np.where(chk['declined'] == 1, 'declined (right call)', 'did NOT decline (weak pick)')
print(chk[['rank', 'content_id', 'outcome']].to_string(index=False))
print(f"\ntop 10 declined: {int(chk.declined.sum())} of 10")

# But are these ten independent bets? Check how concentrated the top of the queue is.
print('\nTop-10 pages by client:', top10['client_id'].value_counts().to_dict())
print('Top-10 distinct page ages (days):', sorted(int(a) for a in top10['content_age_days'].unique()))
for k in (50, 200):
    t = queue.head(k)
    print(f'Top-{k}: {t.client_id.nunique()} clients; largest client supplies {t.client_id.value_counts(normalize=True).iloc[0]:.0%}')

# 'Stale' in the top 10 means something specific: age equals days since update, i.e. never updated since creation.
never = eligible & (df['days_since_last_update'] == df['content_age_days'])
print(f"\nPages never updated since creation: {int(never.sum())} of {int(eligible.sum()):,} eligible | "
      f"share declined {df[never].declined.mean():.3f} vs {df[eligible & ~never].declined.mean():.3f} for the rest")

# Does the rule work equally well on every client? Share declined among pages the rule flags, by client.
flagged = df[hit].groupby('client_id').agg(pages_flagged=('declined', 'size'), share_declined=('declined', 'mean')).round(3)
print('\nPages the rule flags, by client (largest 6 clients; evaluation only):')
print(flagged.sort_values('pages_flagged', ascending=False).head(6))
print(f"All flagged pages: {int(hit.sum()):,} | share declined {df[hit].declined.mean():.3f} | "
      f"all eligible pages {df[eligible].declined.mean():.3f}")

# Leakage check: the columns the rule reads, and the label-derived columns it must not read.
RULE_INPUTS = ['content_age_days', 'impressions_prev_30d', 'days_since_last_update']
BANNED = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
          'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'declined']
assert not set(RULE_INPUTS) & set(BANNED), 'a banned column is a rule input'
print('\nrule inputs:', RULE_INPUTS)
print('none of the banned label-derived or window-overlapping columns is a rule input: OK')

# Receipt: small JSON, safe to commit.
receipt = {'seed': SEED, 'thresholds': {'age_max_days': AGE_MAX, 'vol_lo': VOL_LO, 'vol_hi': VOL_HI},
           'rule_inputs': RULE_INPUTS, 'pages_passing_rule': int(hit.sum()),
           'precision_at_k_test': ev.to_dict(orient='index'),
           'top10_declined': int(chk.declined.sum())}
with open(os.path.join(OUT_DIR, 'w04_baseline.json'), 'w') as f:
    json.dump(receipt, f, indent=2, default=float)
print('wrote work/outputs/w04_baseline.json')

 rank           content_id               outcome
    1 content_c3970ef960d5 declined (right call)
    2 content_69fad7e6c50c declined (right call)
    3 content_7bba5d83959a declined (right call)
    4 content_b08562686d22 declined (right call)
    5 content_9b28cf5ae4f3 declined (right call)
    6 content_452a4e18212c declined (right call)
    7 content_5aa14ed789b2 declined (right call)
    8 content_701f7861d94a declined (right call)
    9 content_434d25702cdb declined (right call)
   10 content_e91ce7f0a0b3 declined (right call)

top 10 declined: 10 of 10

Top-10 pages by client: {'client_7f2253d7e2': 7, 'client_f369cb89fc': 3}
Top-10 distinct page ages (days): [106]
Top-50: 6 clients; largest client supplies 46%
Top-200: 7 clients; largest client supplies 43%

Pages never updated since creation: 103 of 26,612 eligible | share declined 0.864 vs 0.610 for the rest

Pages the rule flags, by client (largest 6 clients; evaluation only):
                   pages_flagged  share_declined


## Data limits

**Named limitation: clients differ a lot, and the queue leans on a few of them.** Among the pages the rule flags, the share that declined runs from 64.7% to 97.1% across the six largest clients. The rule cannot separate a page that is decaying from a client whose whole site moved. The Week 5 model should use client-holdout validation and report results per client.

Other limits. This is one 90-day snapshot, not a time series of the same pages. "Declined" is a 30-day proxy, not proof that an update helps. The 180-day staleness cutoff cannot be tested because almost no page is that stale. The lane is provisional.

# ---- Signal 2 (flag-linked): staleness. Flag logic `stale_visible_page` assumes LONGER since update = riskier.
stale_t = bucket_table(dev, 'days_since_last_update', [-1, 30, 90, 180, 400],
                       ['0-30 days', '31-90 days', '91-180 days', '181+ days'])
print('SIGNAL 2: days since last update, dev clients only (buckets under n=200 do not count)')
print(stale_t)
print('flag assumes: staler pages decline more  ->  VERDICT (raw):', verdict(stale_t, expect='up'))

# Age and staleness are tangled (young pages are often the freshly updated ones). Does staleness still
# matter inside each age band? Compare stale (>= 90 days) vs not, per band.
dev2 = dev.assign(age_band=pd.cut(dev['content_age_days'], [89, 180, 365, 600],
                                  labels=['90-180 days', '181-365 days', '366+ days']),
                  stale=np.where(dev['days_since_last_update'] >= 90, 'stale (>=90d)', 'recent (<90d)'))
within = (dev2.groupby(['age_band', 'stale'], observed=True)
              .agg(n=('declined', 'size'), share_declined=('declined', 'mean')).round(3))
print('\nStaleness inside each age band (dev clients):')
print(within)